In [ ]:
#ncRNA
# substring macthing for ncRNA

import gzip
import csv
from tqdm import tqdm
from itertools import product
import ahocorasick

# =========================
# CONFIG
# =========================
TARGET_FASTA = "target.fasta"
PEPTIDE_CSV  = "final_absent_pep.csv"
DATABASE_FASTA_GZ = "rnacentral_active.fasta.gz"

OUTPUT_TARGET  = "ncRNA_CQWW.csv"
OUTPUT_PEPTIDE = "ncRNA_pent.csv"

DB_BATCH_SIZE = 50

# =========================
# CODON TABLE
# =========================
CODON_TABLE = {
    'A': ['GCT','GCC','GCA','GCG'],
    'C': ['TGT','TGC'],
    'D': ['GAT','GAC'],
    'E': ['GAA','GAG'],
    'F': ['TTT','TTC'],
    'G': ['GGT','GGC','GGA','GGG'],
    'H': ['CAT','CAC'],
    'I': ['ATT','ATC','ATA'],
    'K': ['AAA','AAG'],
    'L': ['TTA','TTG','CTT','CTC','CTA','CTG'],
    'M': ['ATG'],
    'N': ['AAT','AAC'],
    'P': ['CCT','CCC','CCA','CCG'],
    'Q': ['CAA','CAG'],
    'R': ['CGT','CGC','CGA','CGG','AGA','AGG'],
    'S': ['TCT','TCC','TCA','TCG','AGT','AGC'],
    'T': ['ACT','ACC','ACA','ACG'],
    'V': ['GTT','GTC','GTA','GTG'],
    'W': ['TGG'],
    'Y': ['TAT','TAC']
}

# =========================
# LOAD TARGET FASTA
# =========================
def load_target_fasta(filepath):
    queries = {}
    with open(filepath) as f:
        header, seq = None, []
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if header:
                    queries[header] = "".join(seq).upper()
                header = line[1:]
                seq = []
            else:
                seq.append(line)
        if header:
            queries[header] = "".join(seq).upper()
    return queries

# =========================
# LOAD PEPTIDES
# =========================
def load_peptides(filepath):
    peptides = []
    with open(filepath) as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            peptides.append(row[0].strip().upper())
    return peptides

# =========================
# REVERSE TRANSLATION
# =========================
def reverse_translate(peptide):
    codon_lists = [CODON_TABLE[aa] for aa in peptide]
    for c in product(*codon_lists):
        yield "".join(c)

# =========================
# BUILD AUTOMATON
# =========================
def build_automaton(peptides):
    A = ahocorasick.Automaton()

    for pep in tqdm(peptides, desc="Building automaton"):
        for dna in reverse_translate(pep):
            A.add_word(dna, pep)

    A.make_automaton()
    return A

# =========================
# STREAM FASTA.GZ
# =========================
def stream_fasta_gz(filepath):
    with gzip.open(filepath, "rt") as f:
        header, seq = None, []
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if header:
                    yield header, "".join(seq)
                header = line[1:]
                seq = []
            else:
                seq.append(line)
        if header:
            yield header, "".join(seq)

# =========================
# HEADER PARSER
# =========================
def parse_header(header):
    parts = header.strip().split()

    urs_id = parts[0] if len(parts) > 0 else "NA"
    rna_type = parts[1] if len(parts) > 1 else "NA"

    species_count = "NA"
    if "species" in parts:
        try:
            idx = parts.index("species")
            species_count = parts[idx - 1]
        except:
            pass

    return {
        "URS_ID": urs_id,
        "RNA_Type": rna_type,
        "Species_Count": species_count
    }

# =========================
# MAIN
# =========================
def main():

    targets = load_target_fasta(TARGET_FASTA)
    peptides = load_peptides(PEPTIDE_CSV)

    automaton = build_automaton(peptides)

    # WRITE HEADERS
    with open(OUTPUT_TARGET, "w", newline="") as f:
        csv.writer(f).writerow([
            "Target_ID","Target_Sequence",
            "URS_ID","RNA_Type","Species_Count",
            "Hit_Count","Positions"
        ])

    with open(OUTPUT_PEPTIDE, "w", newline="") as f:
        csv.writer(f).writerow([
            "Peptide",
            "Matched_15mer",
            "URS_ID","RNA_Type","Species_Count",
            "Hit_Count","Positions"
        ])

    batch = []

    with open(OUTPUT_TARGET, "a", newline="") as ft, \
         open(OUTPUT_PEPTIDE, "a", newline="") as fp:

        writer_t = csv.writer(ft)
        writer_p = csv.writer(fp)

        for header, seq in tqdm(stream_fasta_gz(DATABASE_FASTA_GZ)):

            batch.append((header, seq.upper()))

            if len(batch) == DB_BATCH_SIZE:

                for header, seq in batch:

                    info = parse_header(header)

                    # =========================
                    # TARGET SEARCH
                    # =========================
                    for tid, tseq in targets.items():
                        positions = []
                        start = 0
                        while True:
                            pos = seq.find(tseq, start)
                            if pos == -1:
                                break
                            positions.append(pos)
                            start = pos + 1

                        if positions:
                            writer_t.writerow([
                                tid, tseq,
                                info["URS_ID"],
                                info["RNA_Type"],
                                info["Species_Count"],
                                len(positions),
                                ",".join(map(str, positions))
                            ])

                    # =========================
                    # PEPTIDE SEARCH
                    # =========================
                    hits = {}

                    for end_pos, pep in automaton.iter(seq):
                        start_pos = end_pos - 15 + 1
                        matched_15mer = seq[start_pos:end_pos + 1]

                        if pep not in hits:
                            hits[pep] = {
                                "positions": [],
                                "matches": []
                            }

                        hits[pep]["positions"].append(start_pos)
                        hits[pep]["matches"].append(matched_15mer)

                    for pep, data in hits.items():
                        writer_p.writerow([
                            pep,
                            ";".join(data["matches"]),
                            info["URS_ID"],
                            info["RNA_Type"],
                            info["Species_Count"],
                            len(data["positions"]),
                            ",".join(map(str, data["positions"]))
                        ])

                batch = []

    print("\nDONE — matched 15-mers added successfully!")

# =========================
if __name__ == "__main__":
    main()


In [ ]:
#promoters
# substring matching for promoters

import pandas as pd
import itertools
import ahocorasick
from tqdm import tqdm
from collections import defaultdict
from Bio.Data import CodonTable

# ==============================
# CONFIG
# ==============================
DNA_FASTA = "promoters.fasta"

PEPTIDE = "CQWW"

OUTPUT_FILE = "promoters_CQWW.csv"

DNA_LENGTH = len(PEPTIDE) * 3

# ==============================
# HEADER PARSER
# ==============================
def parse_header(header):

    parts = header.strip().split("|")

    info = {
        "uniprot_id": parts[0] if len(parts) > 0 else "",
        "nucleotide_id": parts[1] if len(parts) > 1 else "",
        "protein_id": parts[2] if len(parts) > 2 else "",
        "molecule_type": parts[3] if len(parts) > 3 else "",
        "status": parts[4] if len(parts) > 4 else "",
        "uniprot_length": parts[5] if len(parts) > 5 else "",
        "ncbi_length": parts[6] if len(parts) > 6 else "",
        "transl_table": "1",
        "codon_start": "1"
    }

    for p in parts:

        if p.startswith("transl_table="):

            val = p.split("=")[1]

            if val.isdigit():
                info["transl_table"] = val

        elif p.startswith("codon_start="):

            val = p.split("=")[1]

            if val.isdigit():
                info["codon_start"] = val

    return info

# ==============================
# FASTA READER
# ==============================
def read_fasta(file):

    with open(file, "r") as f:

        header = ""
        seq = ""

        for line in f:

            line = line.strip()

            if line.startswith(">"):

                if seq:
                    yield header, seq

                header = line[1:]
                seq = ""

            else:
                seq += line.upper()

        if seq:
            yield header, seq

# ==============================
# GET CODON TABLE
# ==============================
def get_codon_map(table_id):

    try:
        table = CodonTable.unambiguous_dna_by_id[int(table_id)]

    except:
        table = CodonTable.unambiguous_dna_by_id[1]

    codon_map = defaultdict(list)

    for codon, aa in table.forward_table.items():
        codon_map[aa].append(codon)

    return codon_map

# ==============================
# PEPTIDE → DNA VARIANTS
# ==============================
def peptide_to_dna(peptide, codon_map):

    codon_lists = [
        codon_map[aa]
        for aa in peptide
        if aa in codon_map
    ]

    for combo in itertools.product(*codon_lists):

        yield "".join(combo)

# ==============================
# BUILD AUTOMATON
# ==============================
def build_automaton(peptide, codon_map):

    A = ahocorasick.Automaton()

    total_variants = 0

    for dna_seq in peptide_to_dna(peptide, codon_map):

        A.add_word(dna_seq, dna_seq)

        total_variants += 1

    A.make_automaton()

    print(f"\nTotal DNA variants generated: {total_variants}")

    return A

# ==============================
# SAFE SORT
# ==============================
def safe_int(x):

    try:
        return int(x)

    except:
        return 999

# ==============================
# MAIN
# ==============================
def main():

    print(f"\nSearching for peptide: {PEPTIDE}")

    # ======================================
    # GROUP FASTA BY TRANSLATION TABLE
    # ======================================
    fasta_groups = defaultdict(list)

    print("\nGrouping FASTA by transl_table...")

    for header, seq in read_fasta(DNA_FASTA):

        info = parse_header(header)

        fasta_groups[
            info["transl_table"]
        ].append((header, seq, info))

    results = []

    # ======================================
    # PROCESS EACH TABLE
    # ======================================
    for tt in sorted(fasta_groups.keys(), key=safe_int):

        entries = fasta_groups[tt]

        print("\n==============================")
        print(f"Processing transl_table = {tt}")
        print(f"Sequences = {len(entries)}")
        print("==============================")

        codon_map = get_codon_map(tt)

        print("\nBuilding automaton...")

        automaton = build_automaton(
            PEPTIDE,
            codon_map
        )

        print("\nScanning sequences...")

        for header, seq, header_info in tqdm(
            entries,
            desc=f"Table {tt}"
        ):

            # UNIQUE POSITIONS ONLY
            positions = set()

            # UNIQUE DNA VARIANTS
            dna_matches = set()

            # ==========================
            # SEARCH
            # ==========================
            for end_idx, dna_seq in automaton.iter(seq):

                start_idx = (
                    end_idx - DNA_LENGTH + 1
                )

                positions.add(start_idx)

                dna_matches.add(dna_seq)

            # ==========================
            # SAVE HITS
            # ==========================
            if len(positions) > 0:

                results.append({

                    "uniprot_id":
                        header_info["uniprot_id"],

                    "nucleotide_id":
                        header_info["nucleotide_id"],

                    "protein_id":
                        header_info["protein_id"],

                    "molecule_type":
                        header_info["molecule_type"],

                    "status":
                        header_info["status"],

                    "uniprot_length":
                        header_info["uniprot_length"],

                    "ncbi_length":
                        header_info["ncbi_length"],

                    "transl_table":
                        header_info["transl_table"],

                    "codon_start":
                        header_info["codon_start"],

                    "peptide":
                        PEPTIDE,

                    "matched_15mer":
                        ";".join(sorted(dna_matches)),

                    "num_hits":
                        len(positions),

                    "positions":
                        ";".join(
                            map(
                                str,
                                sorted(positions)
                            )
                        ),

                    "cds_sequence":
                        seq
                })

    # ======================================
    # SAVE OUTPUT
    # ======================================
    print("\nSaving output...")

    output_df = pd.DataFrame(results)

    column_order = [

        "uniprot_id",
        "nucleotide_id",
        "protein_id",

        "molecule_type",
        "status",

        "uniprot_length",
        "ncbi_length",

        "transl_table",
        "codon_start",

        "peptide",

        "matched_15mer",

        "num_hits",

        "positions",

        "cds_sequence"
    ]

    output_df = output_df[column_order]

    output_df.to_csv(
        OUTPUT_FILE,
        index=False
    )

    # ======================================
    # DONE
    # ======================================
    print("\n==============================")
    print("DONE!")
    print("==============================")

    print(f"\nTotal sequences with hits: {len(output_df)}")

    print(f"\nSaved output:")
    print(OUTPUT_FILE)

    if len(output_df) > 0:

        print("\nFirst 5 rows:\n")

        print(
            output_df[[
                "peptide",
                "matched_15mer",
                "num_hits",
                "positions"
            ]].head()
        )

# ==============================
# RUN
# ==============================
if __name__ == "__main__":

    main()